# DATT — Chạy project trên Colab T4

Chạy lần lượt **Cell 1 → 13** trên runtime hiện tại. Không reset runtime. Source và model hiện có trong /content/DATT; bộ Torch CUDA 12.6 đã được kiểm chứng.

*Tùy chọn Drive Cache:* Để tối ưu hóa và tăng tốc cho các phiên chạy sau bằng Google Drive (cache Git mirror, pip wheels, models), bạn có thể chạy cell tiện ích 
otebooks/colab_drive_cache_cell.py trước khi chạy Cell 1.

1. **Cell 1–3:** kiểm tra phép tính GPU, source và dependencies.
2. **Cell 4:** tự đọc /content/DATT/.env hoặc Colab Secrets. Nếu thiếu, nhập cấu hình qua ô nhập ẩn. Không dán mật khẩu vào code. Cần DATT_DATABASE_URL, SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY, DATT_STORAGE_BUCKET; backend mặc định supabase. SMTP dùng Secrets có sẵn hoặc ô nhập ẩn. Nếu dùng .env, điền đúng hoặc bỏ các dòng giá trị rỗng trước khi chạy.
3. **Cell 5–10:** kiểm tra PostgreSQL/pgvector/Storage, model, áp dụng migration hiện có và khởi động dịch vụ. Storage preflight tạo rồi xóa object thử riêng.
4. **Cell 11:** mở link ứng dụng được in ra. **Cell 12–13:** kiểm tra API và trạng thái startup.

Nếu cell lỗi: sửa nguyên nhân, chạy lại từ **Cell 1**. Không chạy vượt cell lỗi. Notebook dừng khi thiếu cấu hình, source/model hoặc dependency không tương thích. Runtime mới cần nạp lại source/model; notebook này dùng source đang triển khai, không tự clone repository.

SYSTEM_READY=YES chỉ xác nhận startup/smoke check thành công, chưa bao gồm Real Camera Acceptance. Cấu hình nhập ẩn chỉ lưu trong môi trường runtime; không ghi vào notebook hay .env.


In [ ]:
# CELL 1 — Provision the supported CUDA stack, then test the GPU
import os, sys, json, subprocess, socket
from pathlib import Path

ROOT = Path('/content/DATT')
startup = None
runtime = None
print('python=' + sys.version.split()[0])
expected = {'torch': '2.11.0+cu126', 'torchvision': '0.26.0+cu126', 'torchaudio': '2.11.0+cu126'}
version_code = "import importlib.metadata as m,json; names=('torch','torchvision','torchaudio'); print(json.dumps({n:m.version(n) for n in names if m.packages_distributions().get(n)}))"
versions = subprocess.run([sys.executable, '-c', version_code], capture_output=True, text=True, timeout=90)
if versions.returncode:
    raise RuntimeError('Khong doc duoc phien ban Torch trong runtime.')
versions_info = json.loads(versions.stdout)
reuse_existing = False
if versions.returncode == 0:
    try:
        probe_existing = subprocess.run([sys.executable, '-c',
            "import torch,json; x=torch.arange(4,device='cuda'); torch.cuda.synchronize(); print(json.dumps(dict(gpu=torch.cuda.get_device_name(0),cuda=torch.version.cuda,available=torch.cuda.is_available(),torch=torch.__version__,tensor_ok=(x.sum().item()==6))))"],
            capture_output=True, text=True, timeout=90)
        if probe_existing.returncode == 0:
            existing_info = json.loads(probe_existing.stdout)
            reuse_existing = bool(existing_info.get('available') and existing_info.get('tensor_ok'))
    except Exception:
        reuse_existing = False
if json.loads(versions.stdout) != expected and not reuse_existing:
    # Do not replace native libraries underneath a running kernel/service.
    if any(n in sys.modules for n in ('torch', 'torchvision', 'torchaudio', 'onnxruntime')):
        raise RuntimeError('Chon Runtime > Restart session, sau do chay lai Cell 1 de cai Torch cu126. Khong Delete runtime.')
    for port in (8000, 8501):
        with socket.socket() as sock:
            sock.settimeout(1)
            if sock.connect_ex(('127.0.0.1', port)) == 0:
                raise RuntimeError('Dung dich vu DATT bang CLI truoc khi doi Torch: port ' + str(port))
    if subprocess.run(['nvidia-smi', '-L'], capture_output=True, timeout=30).returncode:
        raise RuntimeError('Chon GPU T4 trong Runtime > Change runtime type.')
    print('Dang cai Torch CUDA 12.6 tu download.pytorch.org; co the mat vai phut.', flush=True)
    log_path = Path('/content/datt-torch-install.log')
    with log_path.open('w') as log:
        result = subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
            '--only-binary=:all:', '--index-url', 'https://download.pytorch.org/whl/cu126',
            *[n + '==' + v for n, v in expected.items()]], stdout=log, stderr=subprocess.STDOUT, timeout=1200)
    if result.returncode:
        raise RuntimeError('Cai Torch cu126 that bai; xem /content/datt-torch-install.log, sau do chay lai Cell 1.')
probe_code = "import torch,torchvision,torchaudio,json; x=torch.arange(4,device='cuda'); torch.cuda.synchronize(); print(json.dumps(dict(gpu=torch.cuda.get_device_name(0),cuda=torch.version.cuda,available=torch.cuda.is_available(),torch=torch.__version__,torchvision=torchvision.__version__,torchaudio=torchaudio.__version__,tensor_ok=(x.sum().item()==6))))"
probe = subprocess.run([sys.executable, '-c', probe_code], capture_output=True, text=True, timeout=90)
if probe.returncode:
    Path('/content/datt-gpu-probe.log').write_text(probe.stderr)
    raise RuntimeError('GPU smoke test FAIL; xem /content/datt-gpu-probe.log.')
info = json.loads(probe.stdout)
for key in ('gpu', 'torch', 'cuda', 'available', 'tensor_ok'):
    print(str(key) + '=' + str(info[key]))
if not info['available'] or not info['tensor_ok']:
    raise RuntimeError('GPU tensor test FAIL; verify Colab GPU runtime and Torch/CUDA installation.')
print('torch_stack=' + ('REUSED' if any(info[n] != v for n, v in expected.items()) else 'EXPECTED'))
runtime = dict(runtime='colab', gpu=info['gpu'], cuda=info['cuda'], runtime_pid=os.getpid())
print('cell1=PASS')


In [ ]:
# CELL 2 - Update a clean checkout to origin/dev, preserving ignored data
if not isinstance(runtime, dict) or runtime.get('runtime') != 'colab':
    raise RuntimeError('Run Cell 1 successfully first.')
if not (ROOT / '.git').exists():
    raise RuntimeError('/content/DATT must be a Git checkout; restore source without deleting models/data/.env.')
def source_git(*args):
    result = subprocess.run(['git', *args], cwd=ROOT, capture_output=True, text=True, timeout=120)
    if result.returncode:
        raise RuntimeError('Git source update failed; inspect checkout/authentication privately. No reset performed.')
    return result.stdout.strip()
if source_git('status', '--porcelain', '--untracked-files=no'):
    raise RuntimeError('Tracked local changes exist; commit/stash them before updating. No changes discarded.')
source_git('fetch', 'origin', 'dev')
local_sha = source_git('rev-parse', 'HEAD')
remote_sha = source_git('rev-parse', 'origin/dev')
ancestor = subprocess.run(['git', 'merge-base', '--is-ancestor', local_sha, remote_sha], cwd=ROOT)
if ancestor.returncode:
    raise RuntimeError('Checkout diverges from origin/dev; resolve manually. No reset performed.')
if local_sha != remote_sha:
    # Run old CLI in a separate interpreter; it verifies PID/token/creation-time ownership.
    stopped = subprocess.run([sys.executable, str(ROOT / 'scripts/datt.py'), 'stop', '--timeout', '120'],
                             cwd=ROOT, capture_output=True, text=True, timeout=150)
    if stopped.returncode:
        raise RuntimeError('Owned DATT service could not stop safely; source was not changed.')
source_git('switch', 'dev')
source_git('merge', '--ff-only', 'origin/dev')
if source_git('rev-parse', 'HEAD') != remote_sha:
    raise RuntimeError('Source revision mismatch; do not start services.')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
# Startup helpers can be cached when notebook cells are rerun in the same kernel.
import importlib
for name in list(sys.modules):
    if name in ('scripts.colab_startup', 'scripts.colab_install') or name == 'src.ops' or name.startswith('src.ops.'):
        sys.modules.pop(name, None)
importlib.invalidate_caches()
from scripts.colab_startup import Startup
startup = Startup(runtime)
print('source=PASS; branch=dev; commit=' + startup.report['commit'])


In [ ]:
# CELL 3 — Kiem tra va cai dependencies con thieu
# Giu nguyen bo Torch cu126 da kiem chung; dung neu phat hien xung dot.
startup.run(3)


In [ ]:
# CELL 4 ? Upload .env or use Colab Secrets (do not paste credentials in code)
# Run this cell in the current browser session to enable the upload widget.
import contextlib, io, os, shutil
from pathlib import Path
from google.colab import files, userdata
from dotenv import load_dotenv
from scripts.colab_startup import install_uploaded_env
if startup is None or startup.failed or startup.completed != 3:
    raise RuntimeError('Run Cells 1, 2, and 3 successfully before Cell 4.')
with contextlib.redirect_stderr(io.StringIO()):
    load_dotenv(ROOT / '.env', override=True, interpolate=False)
uploaded = {} if (ROOT / '.env').is_file() else files.upload()
env_upload = next((Path(name) for name in uploaded if Path(name).name in ('.env', 'env')), None)
if env_upload is not None:
    env_path = ROOT / '.env'
    install_uploaded_env(env_upload, env_path)
    with contextlib.redirect_stderr(io.StringIO()):
        load_dotenv(env_path, override=True, interpolate=False)
    print('ENV_UPLOAD=PASS')
else:
    print('ENV_UPLOAD=SKIPPED; checking required Colab Secrets')
os.environ.setdefault('DATT_STORAGE_BACKEND', 'supabase')
startup.run(4)


In [ ]:
# CELL 5 — PostgreSQL / pgvector / Supabase Storage
# Kiem tra ket noi va thu upload/read/delete mot object tam rieng.
startup.run(5)


In [ ]:
# CELL 6 - Verify models and GPU inference
import shutil, time, zipfile
cell6_started = time.perf_counter()
adaface = ROOT / 'models/face/adaface_ir50_ms1mv2.onnx'
model_cache = Path('/content/drive/MyDrive/DATT-Colab-cache/models')
if not adaface.is_file() and model_cache.is_dir():
    restored = 0
    for cached in model_cache.rglob('*'):
        if cached.is_file():
            target = ROOT / 'models' / cached.relative_to(model_cache)
            if not target.is_file() or target.stat().st_size != cached.stat().st_size:
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(cached, target)
                restored += 1
    print('MODEL_CACHE=RESTORED (restored ' + str(restored) + ' files)')
if not adaface.is_file():
    from google.colab import files
    print('DRIVE_MODEL_CACHE_MISSING: upload datt-colab-models.zip once')
    uploaded = files.upload()
    archives = [Path(name) for name in uploaded if name.endswith('.zip')]
    if len(archives) != 1:
        raise RuntimeError('Select exactly one production model zip')
    with zipfile.ZipFile(archives[0]) as archive:
        for member in archive.infolist():
            destination = (Path('/content') / member.filename).resolve()
            if not destination.is_relative_to((ROOT / 'models').resolve()):
                raise RuntimeError('Invalid model archive path: ' + member.filename)
        archive.extractall('/content')
    if not adaface.is_file():
        raise RuntimeError('AdaFace model missing after archive extraction')
plate_fetcher = ROOT / 'scripts/fetch_plate_line_model.py'
if plate_fetcher.is_file():
    try:
        subprocess.run([sys.executable, str(plate_fetcher)], cwd=ROOT, timeout=120)
    except Exception as exc:
        print('fetch_plate_line_model notice: ' + type(exc).__name__)
startup.run(6)
if 'save_model_cache' in globals():
    save_model_cache()
print('cell6_seconds=' + str(round(time.perf_counter() - cell6_started, 1)))


In [ ]:
# CELL 7 — Chay migration hien co
# Ap dung migration cua repository; khong reset database.
startup.run(7)


In [ ]:
# CELL 8 — Don tien trinh DATT cu
# Chi dung tien trinh cu da xac minh thuoc DATT; giu backend khoe.
startup.run(8)


In [ ]:
# CELL 9 — Khoi dong DATT bang CLI
# Khoi dong che do GPU va cac dich vu do CLI quan ly.
startup.run(9)


In [ ]:
# CELL 10 — Kiem tra backend va notification worker
# Chi PASS khi cac dich vu bat buoc deu khoe.
startup.run(10)


In [ ]:
# CELL 11 — Lay duong dan mo ung dung
# Mo URL Colab duoc in ra sau khi health check PASS.
startup.run(11)


In [ ]:
# CELL 12 - Verify live Agent routes/history/validation without LLM calls
startup.run(12)


In [ ]:
# CELL 13 — Tong ket trang thai san sang
# SYSTEM_READY=YES la startup PASS, khong phai Real Camera Acceptance.
startup.run(13)
